# Spike-sort the raw broadband (`.ns5`)

The `.ns5` is the raw **broadband** stream (~30 kHz) — unlike the 1 kHz `.ns2` LFP, it *can* be spike-sorted.

> **No probe geometry.** These Blackrock files carry no electrode map, so `bio.read_broadband()` attaches a placeholder *independent-channel* probe (channels spaced far apart, so the sorter assumes no adjacency). Per-unit results are valid; cross-channel spatial info is not physical until a real probe map is supplied.

For a full, non-interactive run use `python scripts/run_sorting.py`. This notebook does the same pipeline interactively on a short slice so it finishes quickly.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "scripts"))
import blackrock_io as bio

import spikeinterface.full as si
import spikeinterface.preprocessing as spre
import spikeinterface.sorters as ss
import spikeinterface.widgets as sw

try:
    get_ipython().run_line_magic("matplotlib", "widget")
except Exception:
    get_ipython().run_line_magic("matplotlib", "inline")

print("spikeinterface", si.__version__)
print("installed sorters:", ss.installed_sorters())

## 1. Load the broadband recording (with placeholder probe)

In [ ]:
recording = bio.read_broadband()  # picks the 30 kHz stream + attaches the dummy independent-channel probe
recording

## 2. Preprocess: band-pass + common median reference

In [ ]:
rec_f = spre.bandpass_filter(recording, freq_min=300, freq_max=6000)
rec_f = spre.common_reference(rec_f, reference="global", operator="median")
sw.plot_traces(rec_f, time_range=[0, 2], backend="matplotlib")

## 3. Run a sorter

Sorting the full 132 s takes a few minutes; here we sort the **first 30 s** so the notebook stays snappy. Swap `tridesclous2` for `spykingcircus2` to compare. (For the full recording, run `python scripts/run_sorting.py`.)

In [ ]:
fs = rec_f.get_sampling_frequency()
rec_short = rec_f.frame_slice(start_frame=0, end_frame=int(30 * fs))

sorting = ss.run_sorter(
    "tridesclous2",
    rec_short,
    folder="../outputs/nb_tridesclous2",
    remove_existing_folder=True,
    verbose=True,
)
print(len(sorting.get_unit_ids()), "units:", list(sorting.get_unit_ids()))

In [ ]:
sw.plot_rasters(sorting)

## 4. Quality metrics with a `SortingAnalyzer`

In [ ]:
analyzer = si.create_sorting_analyzer(sorting, rec_short, format="memory")
analyzer.compute(["random_spikes", "waveforms", "templates", "noise_levels"])
analyzer.compute("quality_metrics", metric_names=["firing_rate", "snr", "isi_violation"])
analyzer.get_extension("quality_metrics").get_data().round(3)

## Where to go next

* **Full run + saved outputs** — `python scripts/run_sorting.py` sorts the whole recording and writes the Sorting, a `SortingAnalyzer`, and `quality_metrics.csv` to `outputs/<sorter>/`.
* **Inspect interactively** — open the saved analyzer in `spikeinterface-gui` (`si.plot_sorting_summary(analyzer, backend="spikeinterface_gui")`).
* **Real geometry** — if you obtain the electrode map, build a `probeinterface` `Probe` and `recording.set_probe(...)` instead of the placeholder (see `scripts/blackrock_io.py: attach_dummy_probe`).
* **Compare to the online `.nev` units** — load them with `bio.read_spikes()` and compare against your sorter output.